# Bybit LSTM Training

Обучение LSTM модели

In [ ]:
!pip install -q tensorflow pandas scikit-learn matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import joblib

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DATA_PATH="/content/drive/MyDrive/Bybit_ML_Data/raw_data/"
symbols=["BTCUSDT","ETHUSDT","SOLUSDT","BNBUSDT","XRPUSDT"]
dfs=[]
for symbol in symbols:
    df=pd.read_csv(DATA_PATH+f"klines_{symbol}_60.csv")
    df["symbol"]=symbol
    dfs.append(df)
df_all=pd.concat(dfs,ignore_index=True)
print(f"Total:{len(df_all)}")

In [ ]:
features=["open","high","low","close","volume","rsi","macd","macd_signal","bb_upper","bb_middle","bb_lower","atr","stoch_k","stoch_d","sma_20","sma_50","ema_12","ema_26","hour_sin","hour_cos","day_sin","day_cos","month_sin","month_cos"]
df_all["timestamp"]=pd.to_datetime(df_all["timestamp"])
df_all=df_all.sort_values("timestamp").reset_index(drop=True)
X=df_all[features].values
y=df_all["close"].values

In [ ]:
scaler_X=MinMaxScaler()
scaler_y=MinMaxScaler()
X_scaled=scaler_X.fit_transform(X)
y_scaled=scaler_y.fit_transform(y.reshape(-1,1))

In [ ]:
def create_sequences(X,y,seq_length=60):
    X_seq,y_seq=[],[]
    for i in range(len(X)-seq_length):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i+seq_length])
    return np.array(X_seq),np.array(y_seq)

SEQ_LENGTH=60
X_seq,y_seq=create_sequences(X_scaled,y_scaled,SEQ_LENGTH)
print(f"Sequences:{X_seq.shape}")

In [ ]:
train_size=int(len(X_seq)*0.70)
val_size=int(len(X_seq)*0.15)
X_train=X_seq[:train_size]
y_train=y_seq[:train_size]
X_val=X_seq[train_size:train_size+val_size]
y_val=y_seq[train_size:train_size+val_size]
X_test=X_seq[train_size+val_size:]
y_test=y_seq[train_size+val_size:]
print(f"Train:{len(X_train)},Val:{len(X_val)},Test:{len(X_test)}")

In [ ]:
model=Sequential([
    LSTM(128,return_sequences=True,input_shape=(SEQ_LENGTH,len(features))),
    Dropout(0.2),
    LSTM(64,return_sequences=False),
    Dropout(0.2),
    Dense(32,activation="relu"),
    Dense(1)
])
model.compile(optimizer="adam",loss="mse",metrics=["mae"])
model.summary()

In [ ]:
callbacks=[
    EarlyStopping(monitor="val_loss",patience=15,restore_best_weights=True),
    ModelCheckpoint("best_model.h5",save_best_only=True)
]

In [ ]:
history=model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=100,batch_size=32,callbacks=callbacks)

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history["loss"],label="Train")
plt.plot(history.history["val_loss"],label="Val")
plt.legend()
plt.subplot(1,2,2)
plt.plot(history.history["mae"],label="Train")
plt.plot(history.history["val_mae"],label="Val")
plt.legend()
plt.show()

In [ ]:
y_pred=model.predict(X_test)
y_test_orig=scaler_y.inverse_transform(y_test)
y_pred_orig=scaler_y.inverse_transform(y_pred)
rmse=np.sqrt(mean_squared_error(y_test_orig,y_pred_orig))
mae=mean_absolute_error(y_test_orig,y_pred_orig)
r2=r2_score(y_test_orig,y_pred_orig)
mape=np.mean(np.abs((y_test_orig-y_pred_orig)/y_test_orig))*100
print(f"RMSE:{rmse:.2f},MAE:{mae:.2f},R2:{r2:.4f},MAPE:{mape:.2f}%")

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(y_test_orig[:500],label="Actual")
plt.plot(y_pred_orig[:500],label="Predicted")
plt.legend()
plt.show()

In [ ]:
model.save("bybit_lstm_model.h5")
joblib.dump(scaler_X,"scaler_X.pkl")
joblib.dump(scaler_y,"scaler_y.pkl")
print("Saved!")

In [ ]:
!cp bybit_lstm_model.h5 /content/drive/MyDrive/Bybit_ML_Data/models/
!cp scaler_X.pkl /content/drive/MyDrive/Bybit_ML_Data/models/
!cp scaler_y.pkl /content/drive/MyDrive/Bybit_ML_Data/models/
print("Done!")